In [21]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from urllib.parse import quote

API_KEY = "5c9d733a167dc2b234ae7062a28938d853e92953c9b9998a5071373d5b7eb35e"
END_POINT = "https://apis.data.go.kr/1400377/mtweather"

url = f"{END_POINT}/mountListSearch"
params = {
    "serviceKey": quote(API_KEY, safe=''),
    "numOfRows": 500,
    "pageNo": 1,
    "dataType": "XML"
}

res = requests.get(url, params=params)
root = ET.fromstring(res.text)

stations = []
for item in root.iter("item"):
    obsname = item.findtext("obsname", "")
    obsid = item.findtext("obsid", "")
    localarea = item.findtext("localarea", "")
    stations.append({"stn_id": obsid, "name": obsname, "localarea": localarea})

df_stn = pd.DataFrame(stations)

# 강원도만 (localarea == 10)
gangwon = df_stn[df_stn["localarea"] == "10"].reset_index(drop=True)

# 위경도 추가 (기술문서에서 확인한 값)
# 산악기상 API에는 위경도가 없으니 기상청 AWS 목록에서 가져온 것 사용
print(f"강원도 관측소: {len(gangwon)}개")

강원도 관측소: 153개


In [22]:
import numpy as np

stn_url = "https://apihub.kma.go.kr/api/typ01/url/stn_inf.php"
AUTH_KEY = "EzBXCbK3SQGwVwmyt6kBfQ"

params = {
    "inf": "AWS",
    "stn": "",
    "tm": "202401010000",
    "help": "0",
    "authKey": AUTH_KEY
}

res = requests.get(stn_url, params=params)
lines = res.text.split("\n")

stations = []
for line in lines:
    if line.startswith("#") or line.strip() == "" or "7777" in line:
        continue
    parts = line.split()
    if len(parts) >= 9:
        try:
            stn_id = parts[0]
            lon = float(parts[1])
            lat = float(parts[2])
            name = parts[8]
            stations.append({"stn_id": stn_id, "lat": lat, "lon": lon, "name": name})
        except:
            continue

df_stn = pd.DataFrame(stations)
gangwon = df_stn[
    (df_stn["lat"] >= 37.0) & (df_stn["lat"] <= 38.6) &
    (df_stn["lon"] >= 127.5) & (df_stn["lon"] <= 129.4)
].reset_index(drop=True)

print(f"강원도 AWS 관측소: {len(gangwon)}개")

강원도 AWS 관측소: 106개


In [23]:
import rasterio
import numpy as np

with rasterio.open("../data/raw/NDVI_Gangwon_scaled.tif") as src:
    ndvi = src.read(1)
    print(f"크기: {src.width} x {src.height}")
    print(f"좌표계: {src.crs}")
    print(f"범위: {src.bounds}")
    print(f"NDVI 범위: {ndvi.min()} ~ {ndvi.max()}")

크기: 446 x 358
좌표계: EPSG:4326
범위: BoundingBox(left=127.4978882750837, bottom=36.99711497646249, right=129.50113135867022, top=38.605099335036435)
NDVI 범위: nan ~ nan


C:\Users\ssoos\AppData\Local\Temp\ipykernel_11152\3701520185.py:5: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  ndvi = src.read(1)


In [24]:
with rasterio.open("../data/raw/NDVI_Gangwon_scaled.tif") as src:
    print(f"밴드 수: {src.count}")
    print(f"데이터 타입: {src.dtypes}")
    print(f"nodata 값: {src.nodata}")
    ndvi = src.read(1, masked=True)
    print(f"유효 픽셀 수: {ndvi.count()}")
    print(f"전체 픽셀 수: {ndvi.size}")

밴드 수: 1
데이터 타입: ('float64',)
nodata 값: None
유효 픽셀 수: 159668
전체 픽셀 수: 159668


In [25]:
with rasterio.open("../data/raw/NDVI_Gangwon_scaled.tif") as src:
    ndvi = src.read(1, masked=True)
    print(f"NDVI 범위: {ndvi.min():.4f} ~ {ndvi.max():.4f}")
    print(f"NDVI 평균: {ndvi.mean():.4f}")

NDVI 범위: nan ~ nan
NDVI 평균: nan


In [26]:
with rasterio.open("../data/raw/NDVI_Gangwon_scaled.tif") as src:
    ndvi = src.read(1)
    print(f"전체 값 샘플: {ndvi.flatten()[:20]}")
    print(f"nan 아닌 값 수: {np.sum(~np.isnan(ndvi))}")
    print(f"0 아닌 값 수: {np.sum(ndvi != 0)}")

전체 값 샘플: [0.55191964 0.51102321 0.51102321 0.52660536 0.52660536 0.52660536
 0.50828393 0.50828393 0.38266964 0.38266964 0.44881071 0.44881071
 0.44881071 0.52168214 0.52168214 0.56571071 0.56571071 0.56571071
 0.56486071 0.56486071]
nan 아닌 값 수: 115046
0 아닌 값 수: 159668


C:\Users\ssoos\AppData\Local\Temp\ipykernel_11152\76927471.py:2: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  ndvi = src.read(1)


In [27]:
from rasterio.sample import sample_gen

with rasterio.open("../data/raw/NDVI_Gangwon_scaled.tif") as src:
    # 강원도 AWS 지점 좌표에서 NDVI 추출
    coords = [(row["lon"], row["lat"]) for _, row in gangwon.iterrows()]
    ndvi_values = list(sample_gen(src, coords))
    
gangwon["ndvi_mean"] = [v[0] for v in ndvi_values]

print(gangwon[["stn_id", "name", "lat", "lon", "ndvi_mean"]].head(10))
print(f"\nNDVI 추출 완료: {gangwon['ndvi_mean'].notna().sum()}개 지점")

  stn_id   name       lat        lon  ndvi_mean
0     90     속초  38.25085  128.56473   0.303604
1     92  양양(공)  38.05903  128.66298   0.500627
2     93    북춘천  37.94738  127.75443   0.349661
3    100    대관령  37.67713  128.71834   0.322052
4    101     춘천  37.90262  127.73570   0.307054
5    104    북강릉  37.80456  128.85535   0.513543
6    105     강릉  37.75147  128.89099   0.341318
7    106     동해  37.50709  129.12433   0.513345
8    114     원주  37.33749  127.94659   0.277791
9    121     영월  37.18126  128.45743   0.460852

NDVI 추출 완료: 106개 지점


In [28]:
gangwon.to_csv("../data/processed/gangwon_aws_ndvi.csv", index=False, encoding="utf-8-sig")
print("저장 완료")

저장 완료
